In [1]:
# local
from var_check import name_check
from var_check import time_check
from var_check import url_check
from sqlSrcipts import sql_connect
from sqlSrcipts import sql_engine 
# standard
import os
# third
import pandas as pd
import requests 

In [2]:
name = 'Wolf Pelt'

start = "2026-01-1" #UTC Time
end = "2026-01-25"

condense = 'true'
has_sold = 'true'

limit = '50'
page = '1'

name_check(name)
# time_check(start, end)

def query_date():
    conn = sql_connect()
    cursor = conn.cursor()
    query = "SELECT MAX(created_at) FROM test"
    cursor.execute(query)
    latest_date = cursor.fetchone()[0]
    conn.close()
    return latest_date

def url_page():
    url = (
        f'https://api.darkerdb.com/v1/market'
        f'?key={os.getenv("dark_api_key")}'
        f'&item={name}'
        f'&condense={condense}&has_sold={has_sold}'
        f'&limit={limit}&page={page}'
        f'&from={start}')
    return url

def query_cursor():
    conn = sql_connect()
    cursor = conn.cursor()
    query = "SELECT MAX(cursor) FROM test"
    cursor.execute(query)
    latest_cursor = str(cursor.fetchone()[0])
    conn.close()
    return latest_cursor

def url_cursor():
    url = (
        f'https://api.darkerdb.com/v1/market'
        f'?key={os.getenv("dark_api_key")}'
        f'&item={name}'
        f'&limit={limit}&page={page}'
        f'&condense={condense}&has_sold={has_sold}'
        f'&from={start}&to={end}'
        f"&cursor={latest_cursor}")
    return url
# req = requests.get(url_page())
# req = requests.get(url_cursor())

In [3]:
url_page()


'https://api.darkerdb.com/v1/market?key=meowlin&item=Wolf Pelt&condense=true&has_sold=true&limit=50&page=1&from=2026-01-1'

In [ ]:
# API's cursor pagination not working
# request through cursor
with requests.Session() as ses:
    req_body = []
    json = {"pagination": {"count": int(limit)}}
    latest_cursor = query_cursor()
    
    byte_count, request_count = 0,1    
    while json['pagination']['count'] == 50:
        req = ses.get(url_cursor())
        json = req.json()
        req_body.extend(json['body'])
        latest_cursor = json['body'][0]['cursor'] #sometime req stop comming, rate limited?
          
        print(f'{request_count}', end="\r")
        request_count+=1        
        byte_count += len(req.content)
    df = pd.json_normalize(req_body, sep=',')
    print(f'get-requests = {page} \n bytes = {byte_count}')

In [16]:
# request through pagination
# start = query_date() # for update
with requests.Session() as ses:
    req_body = []
    json = {"pagination": {"count": int(limit)}}
    byte_count = 0    
    while json['pagination']['count'] == 50:
        req = ses.get(url_page())
        json = req.json()
        req_body.extend(json['body'])
        print(f'page {page}', end="\r")
        page = str(int(page) + 1) #sometime req stop comming, rate limited?
        byte_count += len(req.content)
    df = pd.json_normalize(req_body, sep=',')
    print(f'get-requests = {page} \nbytes = {byte_count}')

get-requests = 262 
bytes = 4443582


In [17]:
conn = sql_connect()
cursor = conn.cursor()

df = pd.DataFrame(req_body)

type_map = {
    "int64": "BIGINT",
    "float64": "DOUBLE PRECISION",
    "object": "TEXT",
    "datetime64[ns]": "TIMESTAMP",
    "bool" : "BOOLEAN"    
}
schema = {col: type_map[str(dtype)] for col, dtype in zip(df.columns, df.dtypes)}
columns_dtype = ", ".join(f"{col} {dtype}" for col, dtype in schema.items())
query = f'''
    CREATE TABLE IF NOT EXISTS "MarketSold_{name.replace(" ", "")}" (
    {columns_dtype},
    PRIMARY KEY (cursor)
    );'''
cursor.execute(query)

sql_Col = ', '.join(df.columns)
sql_Placeholder = ", ".join(["%s"] * len(df.columns))
query = f'''
    INSERT INTO "MarketSold_{name.replace(" ", "")}" ({sql_Col}) 
    VALUES ({sql_Placeholder})
    ON CONFLICT (cursor) DO NOTHING;
    '''
rows = [tuple(instance[c] for c in df.columns)for instance in req_body]
cursor.executemany(query,rows)

conn.commit()
conn.close()